# Week 6 Transformer translator

This notebook builds a Transformer model for sequence-to-sequence translation.

**Sections:**
1. Imports & configuration
2. Data loading & exploration
3. Text preprocessing & tokenization
4. Model architecture (embeddings, encoder, decoder)
5. Training
6. Translation examples

In [44]:
import os
import re
import string
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow.strings as tf_strings
from keras.src.layers import TextVectorization
import keras.ops as ops
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")
print(f"GPUs available     : {len(tf.config.list_physical_devices('GPU'))}")

TensorFlow version : 2.21.0
Keras version      : 3.13.2
GPUs available     : 0


In [45]:
TEXT_FILE = "datasets/fin-eng/fin.txt"

with open(TEXT_FILE, encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    english, finnish, *_ = line.split("\t")  # *_ safely discards attribution column
    finnish = "[start] " + finnish + " [end]"
    text_pairs.append((english, finnish))

print(f"Total sentence pairs: {len(text_pairs)}")
print("\nSample pairs:")
for eng, fin in random.sample(text_pairs, 5):
    print(f"  EN: {eng}")
    print(f"  FI: {fin}")
    print()

Total sentence pairs: 73604

Sample pairs:
  EN: Tom is downhearted.
  FI: [start] Tom on alakuloinen. [end]

  EN: We apologize.
  FI: [start] Me olemme pahoillamme. [end]

  EN: I'll be waiting over there.
  FI: [start] Minä odotan tuolla. [end]

  EN: You'd better relax a bit.
  FI: [start] Sinun olisi parempi hieman rentoutua. [end]

  EN: Aren't you afraid to die?
  FI: [start] Etkö pelkää kuolemaa? [end]



## Data Exploration

Before setting model hyperparameters like `max_sequence_length` and `vocab_size`,
we need to understand the data distribution. Key questions:
- How long are the sentences? This determines `sequence_length`.
- How large are the vocabularies? This determines `vocab_size`.

Picking these values blindly wastes capacity or truncates too many sentences.

In [46]:
eng_lengths = [len(eng.split()) for eng, _ in text_pairs]
fin_lengths = [len(fin.split()) for _, fin in text_pairs]

print("=== English ===")
print(f"  Vocab (unique tokens) : {len(set(t for eng, _ in text_pairs for t in eng.split()))}")
print(f"  Mean length           : {np.mean(eng_lengths):.1f} words")
print(f"  95th percentile       : {int(np.percentile(eng_lengths, 95))} words")
print(f"  Max length            : {max(eng_lengths)} words")

print("\n=== Finnish ===")
print(f"  Vocab (unique tokens) : {len(set(t for _, fin in text_pairs for t in fin.split()))}")
print(f"  Mean length           : {np.mean(fin_lengths):.1f} words")
print(f"  95th percentile       : {int(np.percentile(fin_lengths, 95))} words")
print(f"  Max length            : {max(fin_lengths)} words")

=== English ===
  Vocab (unique tokens) : 19118
  Mean length           : 5.9 words
  95th percentile       : 10 words
  Max length            : 65 words

=== Finnish ===
  Vocab (unique tokens) : 50490
  Mean length           : 6.5 words
  95th percentile       : 10 words
  Max length            : 51 words


## Configuration

We define all hyperparameters in one place so they are easy to find and adjust.
Choices are based on the data exploration above rather than blindly copied from
the English-Spanish reference example.

**Vocabulary size:** The raw token count of 50k+ is misleading since it is
case-sensitive and includes punctuation variants. After `TextVectorization`
applies lowercasing and punctuation stripping, the real unique token count will
be lower. However, Finnish is a highly inflected language where each word form
(e.g. *koira, koiran, koiraa, koirassa*) is its own token, so we use **25,000** as a good starting point.

**Sequence length:** The 95th percentile sentence length was 10 words for English
and 8 for Finnish. We use 20 to cover the vast majority of sentences without
excessive padding.

**Embed dim (256):** The dimensionality of the vector space every token is mapped
into. Larger values capture richer representations but increase training time.

**Latent dim (2048):** The inner dimension of the feed-forward network inside each
Transformer block. It expands the representation to let the model compute more
complex transformations before projecting back down to `embed_dim`.

**Attention heads (8):** The number of parallel attention mechanisms. Each head
attends to different aspects of the sequence. `embed_dim` must be divisible by
this value — here 256 / 8 = 32 dimensions per head.

In [47]:
# Data split ratios
VAL_SPLIT  = 0.15  # 15% validation, 15% test, 70% train

# Vocabulary
VOCAB_SIZE = 25000

# Sequence length
SEQUENCE_LENGTH = 20

# Training
BATCH_SIZE = 64
EPOCHS     = 30

# Transformer architecture
EMBED_DIM  = 256   # Dimension of token + positional embeddings
LATENT_DIM = 2048  # Inner dimension of the feed-forward layers
NUM_HEADS  = 8     # Number of attention heads; EMBED_DIM must be divisible by this

print("Configuration:")
print(f"  Vocab size      : {VOCAB_SIZE}")
print(f"  Sequence length : {SEQUENCE_LENGTH}")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Epochs          : {EPOCHS}")
print(f"  Embed dim       : {EMBED_DIM}")
print(f"  Latent dim      : {LATENT_DIM}")
print(f"  Attention heads : {NUM_HEADS}")
print(f"  Embed / heads   : {EMBED_DIM // NUM_HEADS} (key_dim per head)")

Configuration:
  Vocab size      : 25000
  Sequence length : 20
  Batch size      : 64
  Epochs          : 30
  Embed dim       : 256
  Latent dim      : 2048
  Attention heads : 8
  Embed / heads   : 32 (key_dim per head)


## Train / Validation / Test Split

We shuffle the data before splitting to ensure all three sets have a similar
distribution of sentence lengths and topics. The split follows the reference
example: 70% training, 15% validation, 15% test.

The model is trained on the training set, hyperparameters are tuned based on
validation performance, and the test set is only used for final evaluation.

In [48]:
random.shuffle(text_pairs)

num_val_samples   = int(VAL_SPLIT * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples

train_pairs = text_pairs[:num_train_samples]
val_pairs   = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs  = text_pairs[num_train_samples + num_val_samples :]

print(f"Total pairs      : {len(text_pairs)}")
print(f"Training pairs   : {len(train_pairs)}")
print(f"Validation pairs : {len(val_pairs)}")
print(f"Test pairs       : {len(test_pairs)}")

Total pairs      : 73604
Training pairs   : 51524
Validation pairs : 11040
Test pairs       : 11040


## Text Vectorization

We use Keras `TextVectorization` to convert raw strings into integer sequences.
Two separate vectorizers are needed, one for English and one for Finnish. This is because
they have different vocabularies.

Both layers will:
- Lowercase all text
- Strip punctuation (except `[` and `]` which are part of our `[start]`/`[end]` tokens)

The Finnish vectorizer uses `sequence_length + 1` because during training it needs
to produce both the decoder input (tokens 0 to N) and the target (tokens 1 to N+1)
from the same sequence. We slice these apart in the next step.

In [49]:
# Build the set of characters to strip
# We keep [ and ] to preserve [start] and [end] tokens
strip_chars = string.punctuation
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf_strings.lower(input_string)
    return tf_strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")

eng_vectorization = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize=custom_standardization,
)
fin_vectorization = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH + 1,
    standardize=custom_standardization,
)

# Fit only on training data — never on validation or test
train_eng_texts = [pair[0] for pair in train_pairs]
train_fin_texts = [pair[1] for pair in train_pairs]

eng_vectorization.adapt(train_eng_texts)
fin_vectorization.adapt(train_fin_texts)

print(f"English vocab size : {len(eng_vectorization.get_vocabulary())}")
print(f"Finnish vocab size : {len(fin_vectorization.get_vocabulary())}")

English vocab size : 9465
Finnish vocab size : 25000


## Dataset Pipeline

We format the data into `tf.data.Dataset` objects for efficient training.

At each training step the model receives:
- `encoder_inputs`: the vectorized English sentence
- `decoder_inputs`: the Finnish sentence so far (tokens 0 to N)

And tries to predict:
- `targets`: the Finnish sentence shifted by one step (tokens 1 to N+1)

In [50]:
import tensorflow.data as tf_data

def format_dataset(eng, fin):
    eng = eng_vectorization(eng)
    fin = fin_vectorization(fin)
    return (
        {
            "encoder_inputs": eng,
            "decoder_inputs": fin[:, :-1],  # tokens 0 to N
        },
        fin[:, 1:],  # tokens 1 to N+1 (targets)
    )

def make_dataset(pairs):
    eng_texts, fin_texts = zip(*pairs)
    dataset = tf_data.Dataset.from_tensor_slices(
        (list(eng_texts), list(fin_texts))
    )
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.map(format_dataset, num_parallel_calls=tf_data.AUTOTUNE)
    return dataset.cache().shuffle(2048).prefetch(tf_data.AUTOTUNE)

train_ds = make_dataset(train_pairs)
val_ds   = make_dataset(val_pairs)

# Sanity check — verify shapes are what we expect
for inputs, targets in train_ds.take(1):
    print(f'encoder_inputs shape : {inputs["encoder_inputs"].shape}')
    print(f'decoder_inputs shape : {inputs["decoder_inputs"].shape}')
    print(f'targets shape        : {targets.shape}')

encoder_inputs shape : (64, 20)
decoder_inputs shape : (64, 20)
targets shape        : (64, 20)


The dataset pipeline produced the following shapes:
- encoder_inputs shape : (64, 20)
- decoder_inputs shape : (64, 20)
- targets shape        : (64, 20)

All three tensors have the same shape `(batch_size, sequence_length)` — that is,
64 sentence pairs per batch, each padded or truncated to exactly 20 tokens.

- **`encoder_inputs` (64, 20)** — 64 vectorized English sentences, each 20 tokens long.
- **`decoder_inputs` (64, 20)** — 64 Finnish sentences starting from `[start]` up to
  but not including the last token. This is what the decoder sees as context when
  predicting the next word.
- **`targets` (64, 20)** — the same Finnish sentences shifted one step to the left,
  starting from the first real word up to and including `[end]`. This is what the
  model is trained to predict.

For example, if a Finnish sentence vectorizes to `[start, koira, juoksee, end, 0, 0]`,
then `decoder_inputs` sees `[start, koira, juoksee, end, 0]` and `targets` is
`[koira, juoksee, end, 0, 0]`. At each position N the model must predict the token
at position N+1.

## Model Architecture

The Transformer consists of three building blocks:

1. **`PositionalEmbedding`** — maps token indices to vectors and adds positional
   information. Without this, the model has no sense of word order since attention
   is computed between all token pairs simultaneously.

2. **`TransformerEncoder`** — processes the English input. Uses multi-head
   self-attention so every token can attend to every other token in the source
   sentence, followed by a position-wise feed-forward network.

3. **`TransformerDecoder`** — generates the Finnish output one token at a time.
   Has two attention layers:
   - **Masked self-attention** over the Finnish tokens seen so far. The causal
     mask ensures token N can only attend to tokens 0 to N, never the future.
   - **Cross-attention** over the encoder output — this is where the decoder
     reads the English representation.

Each sub-layer uses a residual connection and layer normalisation, which helps
with training stability and gradient flow.

In [51]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        # Propagate padding mask downstream — padding tokens (index 0) are masked out
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "vocab_size"     : self.vocab_size,
            "embed_dim"      : self.embed_dim,
        })
        return config

In [52]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim  = embed_dim
        self.dense_dim  = dense_dim
        self.num_heads  = num_heads
        self.attention  = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm_1     = layers.LayerNormalization()
        self.layernorm_2     = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        # Convert boolean padding mask to the format MultiHeadAttention expects
        padding_mask = ops.cast(mask[:, None, :], dtype="int32") if mask is not None else None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs,
            attention_mask=padding_mask
        )
        proj_input  = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim" : self.embed_dim,
            "dense_dim" : self.dense_dim,
            "num_heads" : self.num_heads,
        })
        return config

In [53]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim  = embed_dim
        self.latent_dim = latent_dim
        self.num_heads  = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential([
            layers.Dense(latent_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm_1      = layers.LayerNormalization()
        self.layernorm_2      = layers.LayerNormalization()
        self.layernorm_3      = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        inputs, encoder_outputs = inputs

        inputs_padding_mask          = mask[0] if mask is not None else None
        encoder_outputs_padding_mask = mask[1] if mask is not None else None

        # 1. Masked self-attention over decoder inputs
        attention_output_1 = self.attention_1(
            query=inputs, value=inputs, key=inputs,
            use_causal_mask=True,          # ← replaces get_causal_attention_mask()
            query_mask=inputs_padding_mask,
        )
        out_1 = self.layernorm_1(inputs + attention_output_1)

        # 2. Cross-attention over encoder output
        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            query_mask=inputs_padding_mask,
            key_mask=encoder_outputs_padding_mask,
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        # 3. Feed-forward network
        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim"  : self.embed_dim,
            "latent_dim" : self.latent_dim,
            "num_heads"  : self.num_heads,
        })
        return config

## Model Assembly

We now connect the three building blocks into a single end-to-end Transformer model.

The data flow is:
1. English tokens → `PositionalEmbedding` → `TransformerEncoder` → encoder output
2. Finnish tokens → `PositionalEmbedding` → `TransformerDecoder` (+ encoder output) → decoded representation
3. Decoded representation → `Dropout` → `Dense(vocab_size, softmax)` → probability distribution over Finnish vocabulary

Dropout is applied before the final layer as a regularization measure to reduce
overfitting. The output Dense layer produces one probability distribution per
time step, over the entire Finnish vocabulary.

In [54]:
encoder_inputs     = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x                  = PositionalEmbedding(SEQUENCE_LENGTH, VOCAB_SIZE, EMBED_DIM)(encoder_inputs)
encoder_outputs    = TransformerEncoder(EMBED_DIM, LATENT_DIM, NUM_HEADS)(x)

decoder_inputs     = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
x                  = PositionalEmbedding(SEQUENCE_LENGTH, VOCAB_SIZE, EMBED_DIM)(decoder_inputs)
x                  = TransformerDecoder(EMBED_DIM, LATENT_DIM, NUM_HEADS)([x, encoder_outputs])
x                  = layers.Dropout(0.5)(x)
decoder_outputs    = layers.Dense(VOCAB_SIZE, activation="softmax")(x)

transformer = keras.Model(
    inputs={"encoder_inputs": encoder_inputs, "decoder_inputs": decoder_inputs},
    outputs=decoder_outputs,
    name="transformer",
)

transformer.summary()

Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  6,405,120 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  6,405,120 │ decoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 256) │  3,155,456 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decode… │ (None, None, 256) │  5,259,520 │ positional_embed… │
│ (TransformerDecode… │                   │            │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, None, 256) │          0 │ transformer_deco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, None,      │  6,425,000 │ dropout_4[0][0]   │
│                     │ 25000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 27,650,216 (105.48 MB)

 Trainable params: 27,650,216 (105.48 MB)

 Non-trainable params: 0 (0.00 B)

## Training

We compile the model with the `RMSprop` optimizer and sparse categorical
crossentropy loss. The `ignore_class=0` argument tells the loss function to
ignore padding tokens (index 0) when computing the loss — we should not penalize
the model for not predicting padding correctly.

In [55]:
transformer.compile(
    optimizer=keras.optimizers.RMSprop(),
    loss=keras.losses.SparseCategoricalCrossentropy(ignore_class=0),
    metrics=["accuracy"],
)

history = transformer.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            filepath="transformer_best.keras",
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
        )
    ]
)

Epoch 1/30


W0000 00:00:1777373977.184444   73630 cpu_allocator_impl.cc:82] Allocation of 128000000 exceeds 10% of free system memory.
W0000 00:00:1777373977.488103   73634 cpu_allocator_impl.cc:82] Allocation of 128000000 exceeds 10% of free system memory.
W0000 00:00:1777373977.494340   73631 cpu_allocator_impl.cc:82] Allocation of 128000000 exceeds 10% of free system memory.
W0000 00:00:1777373977.538867   73634 cpu_allocator_impl.cc:82] Allocation of 128000000 exceeds 10% of free system memory.
W0000 00:00:1777373977.605097   73634 cpu_allocator_impl.cc:82] Allocation of 128000000 exceeds 10% of free system memory.


 12/806 ━━━━━━━━━━━━━━━━━━━━ 44:02 3s/step - accuracy: 0.1106 - loss: 9.4518

KeyboardInterrupt: 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history["accuracy"],     label="Train accuracy")
ax1.plot(history.history["val_accuracy"], label="Val accuracy")
ax1.set_title("Accuracy over epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history.history["loss"],     label="Train loss")
ax2.plot(history.history["val_loss"], label="Val loss")
ax2.set_title("Loss over epochs")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.suptitle("Transformer Training History", fontsize=14)
plt.tight_layout()
plt.show()

## Translation Examples

We use greedy decoding to translate sentences from the test set. At each step
the model predicts a probability distribution over the Finnish vocabulary and
we pick the highest probability token. The process repeats until the model
produces an `[end]` token or we hit the maximum length.

In [ ]:
fin_vocab        = fin_vectorization.get_vocabulary()
fin_index_lookup = dict(zip(range(len(fin_vocab)), fin_vocab))

def decode_sequence(input_sentence):
    tokenized_input   = eng_vectorization([input_sentence])
    decoded_sentence  = "[start]"

    for i in range(SEQUENCE_LENGTH):
        tokenized_target = fin_vectorization([decoded_sentence])[:, :-1]
        predictions      = transformer(
            {
                "encoder_inputs": tokenized_input,
                "decoder_inputs": tokenized_target,
            }
        )
        sampled_token_index = ops.convert_to_numpy(
            ops.argmax(predictions[0, i, :])
        ).item(0)
        sampled_token    = fin_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break

    return decoded_sentence

print("Translation examples from test set:\n")
for input_sentence, correct_finnish in random.sample(test_pairs, 10):
    translated = decode_sequence(input_sentence)
    print(f"  English  : {input_sentence}")
    print(f"  Expected : {correct_finnish}")
    print(f"  Got      : {translated}")
    print()